## **Unified Multi-Source Cleaning Pipeline - Tik Tok Datasets**

In [3]:
# =========================================================
# Enhanced Unified Multi-Source Cleaning Pipeline (OPTIMIZED)
# =========================================================

import os
import re
import glob
import pandas as pd
from collections import Counter

# =========================================================
# 1) إعدادات
# =========================================================

ROOT_FOLDER = r"C:\Users\aws12\Desktop\Proccesing Datsets ( Tik Tok & Youtube )\الأماكن السياحية في منطقة نجران - Tik Tok"
OUTPUT_FOLDER = r"C:\Users\aws12\Desktop\Proccesing Datsets ( Tik Tok & Youtube )\بيانات منطقة نجران - بعد المعالجة"

TEXT_COLUMN_CANDIDATES = ["Text", "text", "Text_Orig", "comment", "Comment", "comments"]

MIN_WORDS_USEFUL = 3
MIN_ANALYSIS_WORDS = 4
MIN_QUALITY_SCORE = 5
MIN_STRENGTH_SCORE = 6

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# =========================================================
# 2) القوائم
# =========================================================

STOPWORDS = {
    "في", "من", "على", "عن", "الى", "إلى", "او", "أو", "ثم", "لكن", "بس",
    "هذا", "هذه", "هذي", "هنا", "هناك", "هو", "هي", "هم", "انا", "احنا",
    "كان", "كانت", "يكون", "مو", "مش", "مرة", "مره", "جدا", "جداً"
}

NOISE_WORDS = {
    "يفوز", "واو", "ابداع", "فخم", "راقي", "ونعم", "خرافي", "يازينه",
    "حلو", "حلوه", "حلوة", "روعه", "رهيب", "جميل", "جميله", "جميلة",
    "ممتاز", "ابداااع", "يجنن"
}

VERY_SHORT_OPINIONS = {
    "جميل", "جميله", "جميلة", "رائع", "روعه", "رهيب", "حلو", "حلوه", "حلوة",
    "ممتاز", "يفوز", "فخم", "راقي", "ممتع"
}

AD_WORDS = {
    "واتساب", "تواصل", "احجز", "حجز", "تابعني", "عرض", "عروض", "خصم",
    "للتواصل", "سناب", "انستا", "حسابي", "حسابنا", "للحجز", "اعلان", "إعلان"
}

QUESTION_WORDS = {
    "وين", "كيف", "كم", "هل", "وش", "ايش", "ليش", "متى", "فين", "لوسمحت", "ممكن"
}

INTENT_WORDS = {
    "بروح", "بزوره", "بزورها", "ناوي", "ودي", "ابغى", "ابغا", "لازم",
    "بنروح", "بجي", "اضفته", "اضفتها", "نفسي", "ودي اروح", "ودي ازوره"
}

OPINION_WORDS = {
    "جميل", "جميلة", "رائع", "روعه", "ممتاز", "سيء", "سيئ", "زحمة", "هادئ",
    "هادي", "نظيف", "وصخ", "غالي", "رخيص", "يستاهل", "مايستاهل", "حلو",
    "مزعج", "مريح", "فخم", "بارد", "حر", "حار", "ممتع", "يفشل", "رهيب",
    "كبير", "صغير", "هادية", "ممتعه", "جميله", "رايق", "رايقه"
}

PLACE_PREFIXES = {
    "قرية", "منتزه", "حديقة", "جبل", "وادي", "شاطئ", "كورنيش", "ممشى",
    "بوليفارد", "متحف", "اكواخ", "كوخ", "مطل", "عين", "سد", "شلال",
    "غابة", "مزرعة", "تلفريك", "بحيرة", "محمية", "واجهة", "جزيرة",
    "قلعة", "حصن", "قصر"
}

GENERIC_PLACE = {
    "فندق", "فنادق", "شقق", "شقة", "منتجع", "منتجعات", "مكان", "المكان",
    "مدينة", "مدينه", "منطقة", "منطقه", "موقع", "معلم", "مزرعة", "حديقة", "منتزه"
}

REGION_CITY_WORDS = {
    "الرياض", "جده", "جدة", "مكه", "مكة", "المدينة", "المدينه", "الطائف", "الطايف",
    "ابها", "أبها", "الباحه", "الباحة", "جازان", "تبوك", "نجران", "حائل", "حايل",
    "القصيم", "الخبر", "الدمام", "الاحساء", "الأحساء", "ينبع", "العلا", "الدرعية",
    "خميس", "مشيط", "المندق", "بلجرشي", "الشفا", "الهدا", "عسير", "السعودية", "المملكة"
}

# =========================================================
# 3) تنظيف النص
# =========================================================

def normalize_arabic(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # إزالة روابط ومنشن وهاشتاق
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[@#]\S+", " ", text)

    # إزالة التشكيل
    text = re.sub(r"[\u0617-\u061A\u064B-\u0652]", "", text)

    # توحيد بعض الحروف
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)

    # حذف الإنجليزي والأرقام والرموز
    text = re.sub(r"[A-Za-z0-9]", " ", text)
    text = re.sub(r"[^\u0600-\u06FF\s؟!]", " ", text)

    # تقليل التكرار
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)

    # توحيد المسافات
    text = re.sub(r"\s+", " ", text).strip()

    return text

def tokenize(text):
    return [w for w in normalize_arabic(text).split() if len(w.strip()) > 0]

def clean_text(text):
    words = tokenize(text)
    words = [w for w in words if w not in STOPWORDS]
    return " ".join(words).strip()

# =========================================================
# 4) دوال مساعدة للتصنيف
# =========================================================

def is_noise(text):
    words = tokenize(text)
    if len(words) == 0:
        return True
    if len(words) <= 1:
        return True
    if all(w in NOISE_WORDS for w in words):
        return True
    return False

def is_ad(text):
    t = normalize_arabic(text)
    return any(w in t for w in AD_WORDS)

def is_question(text):
    t = normalize_arabic(text)
    return "؟" in t or any(w in t.split() for w in QUESTION_WORDS)

def is_intent(text):
    t = normalize_arabic(text)
    return any(w in t for w in INTENT_WORDS)

def is_opinion(text):
    t = normalize_arabic(text)
    return any(w in t for w in OPINION_WORDS)

def is_generic_only(words):
    return len(words) > 0 and all(w in GENERIC_PLACE or w in REGION_CITY_WORDS for w in words)

def contains_place(text):
    words = tokenize(text)

    if len(words) < 2:
        return False

    # يبدأ ببادئة مكان + كلمة بعدها
    if words[0] in PLACE_PREFIXES and len(words) >= 2:
        # لا نقبل لو كانت الكلمة التالية عامة جدًا
        if len(words) >= 2 and words[1] not in GENERIC_PLACE:
            return True

    # عبارات قصيرة قد تكون اسم مكان فعلي
    if 2 <= len(words) <= 4:
        if not any(w in GENERIC_PLACE for w in words):
            if not any(w in QUESTION_WORDS for w in words):
                if not any(w in AD_WORDS for w in words):
                    return True

    return False

def valid_place_only(text):
    words = tokenize(text)

    if len(words) < 2:
        return False

    if is_generic_only(words):
        return False

    # رفض إذا كانت كلها كلمات عامة
    if all(w in GENERIC_PLACE for w in words):
        return False

    # رفض إذا احتوت على كلمات مكان عامة فقط
    if any(w in {"فندق", "فنادق", "شقق", "شقة", "منتجع", "منتجعات"} for w in words):
        return False

    return True

# =========================================================
# 5) تصنيف التعليق
# =========================================================

def classify_comment(text):
    t = normalize_arabic(text)

    if not t:
        return "noise"

    if is_ad(t):
        return "ad"

    if is_noise(t):
        return "noise"

    place_flag = contains_place(t) and valid_place_only(t)
    opinion_flag = is_opinion(t)

    if is_question(t):
        return "question"

    if is_intent(t):
        return "intent"

    if place_flag and opinion_flag:
        return "place_opinion"

    if opinion_flag:
        return "opinion"

    if place_flag:
        return "place_only"

    return "noise"

# =========================================================
# 6) تقييم الجودة والقوة
# =========================================================

def quality_score(text):
    t = normalize_arabic(text)
    words = t.split()

    score = min(len(words), 10)

    if is_opinion(t):
        score += 3
    if contains_place(t):
        score += 2
    if is_question(t):
        score += 1
    if is_intent(t):
        score += 1
    if is_ad(t):
        score -= 5
    if is_noise(t):
        score -= 4

    return score

def is_useful_text(text):
    words = tokenize(text)

    if len(words) < MIN_WORDS_USEFUL:
        return False

    short_words = sum(len(w) <= 2 for w in words)
    if short_words > len(words) / 2:
        return False

    return True

def comment_strength(text):
    t = normalize_arabic(text)
    words = t.split()

    score = len(words)

    # كلمات تحمل معلومة أدق
    informative_words = {
        "زحمه", "زحمة", "نظيف", "وصخ", "غالي", "رخيص", "بارد", "حار", "هادئ",
        "هادي", "مريح", "مزعج", "جميل", "رائع", "سيء", "ممتع", "يستاهل", "يفشل"
    }

    if any(w in t for w in informative_words):
        score += 5

    if len(words) >= 6:
        score += 3

    return score

# =========================================================
# 7) استخراج الأماكن
# =========================================================

def extract_places(text):
    words = tokenize(text)
    places = []

    for i, w in enumerate(words):
        if w in PLACE_PREFIXES:
            phrase = " ".join(words[i:i+4]).strip()
            phrase_words = phrase.split()

            # إزالة النهاية لو كانت مدينة/منطقة
            while phrase_words and phrase_words[-1] in REGION_CITY_WORDS:
                phrase_words.pop()

            phrase = " ".join(phrase_words).strip()

            if len(phrase.split()) >= 2:
                # رفض الأماكن العامة جدًا
                if not any(x in {"فندق", "فنادق", "شقق", "شقة", "منتجع"} for x in phrase.split()):
                    places.append(phrase)

    return places

# =========================================================
# 8) قراءة الملف
# =========================================================

def load_file(file_path):
    if file_path.lower().endswith(".csv"):
        try:
            return pd.read_csv(file_path)
        except UnicodeDecodeError:
            return pd.read_csv(file_path, encoding="utf-8-sig")
    elif file_path.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(file_path)
    else:
        raise ValueError("Unsupported file type")

def detect_text_column(df):
    for col in TEXT_COLUMN_CANDIDATES:
        if col in df.columns:
            return col
    return None

# =========================================================
# 9) معالجة ملف
# =========================================================

def process_file(file_path):
    df = load_file(file_path)

    text_col = detect_text_column(df)
    if text_col is None:
        return None

    df["Text_Normalized"] = df[text_col].apply(normalize_arabic)
    df["Text_Cleaned"] = df[text_col].apply(clean_text)
    df["comment_type"] = df[text_col].apply(classify_comment)
    df["quality_score"] = df[text_col].apply(quality_score)
    df["is_useful"] = df[text_col].apply(is_useful_text)
    df["comment_strength"] = df[text_col].apply(comment_strength)
    df["places"] = df[text_col].apply(extract_places)

    # بيانات التحليل الأولية
    df_analysis = df[
        (df["comment_type"].isin(["opinion", "place_opinion"])) &
        (df["quality_score"] >= MIN_QUALITY_SCORE) &
        (df["is_useful"] == True)
    ].copy()

    # فلتر إضافي نهائي
    df_analysis = df_analysis[
        (df_analysis["Text_Cleaned"].str.split().str.len() >= MIN_ANALYSIS_WORDS) &
        (~df_analysis["Text_Cleaned"].isin(VERY_SHORT_OPINIONS))
    ].copy()

    # فلتر القوة
    df_analysis = df_analysis[
        (df_analysis["comment_strength"] >= MIN_STRENGTH_SCORE)
    ].copy()

    # استخراج أكثر مكان
    counter = Counter()
    for p in df["places"]:
        if isinstance(p, list):
            counter.update(p)

    top_place = counter.most_common(1)[0][0] if counter else None

    # الحفظ
    name = os.path.splitext(os.path.basename(file_path))[0]

    cleaned_path = os.path.join(OUTPUT_FOLDER, f"{name}_cleaned.xlsx")
    analysis_path = os.path.join(OUTPUT_FOLDER, f"{name}_analysis.xlsx")

    df.to_excel(cleaned_path, index=False)
    df_analysis.to_excel(analysis_path, index=False)

    return {
        "file": name,
        "rows": len(df),
        "analysis_rows": len(df_analysis),
        "top_place": top_place,
        "cleaned_file": cleaned_path,
        "analysis_file": analysis_path
    }

# =========================================================
# 10) تشغيل على جميع الملفات
# =========================================================

def run_all():
    files = glob.glob(os.path.join(ROOT_FOLDER, "**", "*.xlsx"), recursive=True)
    files += glob.glob(os.path.join(ROOT_FOLDER, "**", "*.xls"), recursive=True)
    files += glob.glob(os.path.join(ROOT_FOLDER, "**", "*.csv"), recursive=True)

    results = []

    for f in files:
        print("Processing:", f)
        try:
            res = process_file(f)
            if res:
                results.append(res)
        except Exception as e:
            results.append({
                "file": os.path.basename(f),
                "rows": 0,
                "analysis_rows": 0,
                "top_place": None,
                "cleaned_file": None,
                "analysis_file": None,
                "error": str(e)
            })

    summary = pd.DataFrame(results)
    summary.to_excel(os.path.join(OUTPUT_FOLDER, "summary.xlsx"), index=False)

    print("Finished 🚀")

# =========================================================
# تشغيل
# =========================================================

run_all()

Processing: C:\Users\aws12\Desktop\Proccesing Datsets ( Tik Tok & Youtube )\الأماكن السياحية في منطقة نجران - Tik Tok\Video comments 10_textready.xlsx
Processing: C:\Users\aws12\Desktop\Proccesing Datsets ( Tik Tok & Youtube )\الأماكن السياحية في منطقة نجران - Tik Tok\Video comments 11_textready.xlsx
Processing: C:\Users\aws12\Desktop\Proccesing Datsets ( Tik Tok & Youtube )\الأماكن السياحية في منطقة نجران - Tik Tok\Video comments 12_textready.xlsx
Processing: C:\Users\aws12\Desktop\Proccesing Datsets ( Tik Tok & Youtube )\الأماكن السياحية في منطقة نجران - Tik Tok\Video comments 1_textready.xlsx
Processing: C:\Users\aws12\Desktop\Proccesing Datsets ( Tik Tok & Youtube )\الأماكن السياحية في منطقة نجران - Tik Tok\Video comments 2_textready.xlsx
Processing: C:\Users\aws12\Desktop\Proccesing Datsets ( Tik Tok & Youtube )\الأماكن السياحية في منطقة نجران - Tik Tok\Video comments 3_textready.xlsx
Processing: C:\Users\aws12\Desktop\Proccesing Datsets ( Tik Tok & Youtube )\الأماكن السياحية في م

In [6]:
df = pd.read_excel(r"C:\Users\aws12\Desktop\Proccesing Datsets ( Tik Tok & Youtube )\Datasets After cleaning 3\Video comments 2_textready_cleaned.xlsx")

df

,text,diggCount,replyCommentTotal,createTimeISO,videoWebUrl,cid,Text_Orig,Text,Emoji_List,Emoji_Count,...,Is_Short_ML,Sentiment,Stars,Text_Normalized,Text_Cleaned,comment_type,quality_score,is_useful,comment_strength,places
0,اجواء المكان تفتح النفس,2,0.0,2025-07-23T10:35:49.000Z,https://www.tiktok.com/@h_442b/video/752995151...,7530224185092654080,اجواء المكان تفتح النفس,اجواء المكان تفتح النفس,[],0,...,False,positive,5,اجواء المكان تفتح النفس,اجواء المكان تفتح النفس,noise,4,True,4,[]
1,اغلب هذه المواقع في مدينة الجمال … المندق,6,1.0,2025-07-27T21:21:58.000Z,https://www.tiktok.com/@h_442b/video/752995151...,7531874982738347008,اغلب هذه المواقع في مدينة الجمال … المندق,اغلب هذه المواقع في مدينة الجمال … المندق,[],0,...,False,neutral,3,اغلب هذه المواقع في مدينه الجمال المندق,اغلب المواقع مدينه الجمال المندق,noise,7,True,10,[]
2,حبيييت ماتوقعت هالجمال,3,0.0,2025-07-22T23:05:03.000Z,https://www.tiktok.com/@h_442b/video/752995151...,7530046143598871552,حبيييت ماتوقعت هالجمال,حبيييت ماتوقعت هالجمال,[],0,...,False,positive,5,حبييت ماتوقعت هالجمال,حبييت ماتوقعت هالجمال,place_only,5,True,3,[]
3,تهبل الباحه وهذا وقتها,4,1.0,2025-07-23T01:41:02.000Z,https://www.tiktok.com/@h_442b/video/752995151...,7530086369570374656,تهبل الباحه وهذا وقتها,تهبل الباحه وهذا وقتها,[],0,...,False,positive,5,تهبل الباحه وهذا وقتها,تهبل الباحه وهذا وقتها,place_only,6,True,4,[]
4,المكان يجنن مره,3,0.0,2025-07-22T23:52:41.000Z,https://www.tiktok.com/@h_442b/video/752995151...,7530058373028709376,المكان يجنن مره,المكان يجنن مره,[],0,...,False,positive,5,المكان يجنن مره,المكان يجنن,noise,3,True,3,[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
154,غبار ‍♂️,0,0.0,2025-08-08T15:30:38.000Z,https://www.tiktok.com/@h_442b/video/752995151...,7536237528114856960,غبار ‍♂️,غبار ‍♂️,['♂'],1,...,False,negative,1,غبار,غبار,noise,-3,False,1,[]
155,شقق مفروشه بالباحه جديده للإيجار اليومي والشهر...,0,0.0,2025-07-30T18:13:00.000Z,https://www.tiktok.com/@h_442b/video/752995151...,7532939566178960384,شقق مفروشه بالباحه جديده للإيجار اليومي والشهر...,شقق مفروشه بالباحه جديده للإيجار اليومي والشهر...,[],0,...,False,neutral,3,شقق مفروشه بالباحه جديده للايجار اليومي والشهر...,شقق مفروشه بالباحه جديده للايجار اليومي والشهر...,noise,10,True,16,[]
156,باقي شوي على الأمطار لاتستعجلوا,0,0.0,2025-07-25T12:16:02.000Z,https://www.tiktok.com/@h_442b/video/752995151...,7530992184141628416,باقي شوي على الأمطار لاتستعجلوا,باقي شوي على الأمطار لاتستعجلوا,[],0,...,False,neutral,3,باقي شوي علي الامطار لاتستعجلوا,باقي شوي علي الامطار لاتستعجلوا,noise,5,True,5,[]
157,@Abo Amjd1445: سيارة عائلية باترول في الباحة ل...,0,0.0,2025-07-27T11:42:08.000Z,https://www.tiktok.com/@h_442b/video/752995151...,7531725611766612992,@Abo Amjd1445: سيارة عائلية باترول في الباحة ل...,@Abo Amjd1445: سيارة عائلية باترول في الباحة ل...,[],0,...,False,neutral,3,سياره عايليه باترول في الباحه للايجار اليومي و...,سياره عايليه باترول الباحه للايجار اليومي ومرح...,noise,10,True,14,[]


## **YouTube Comments Cleaning & Classification Pipeline**

In [8]:
# =========================================================
# Enhanced YouTube Comments Cleaning & Classification Pipeline
# Optimized for Saudi Tourism Comments
# =========================================================

import os
import re
import glob
import pandas as pd
from collections import Counter

# =========================================================
# 1) إعدادات
# =========================================================

ROOT_FOLDER = r"C:\Users\aws12\Desktop\Proccesing Datsets ( Tik Tok & Youtube )\YouTube Datasets - Najran"   # عدل المسار
OUTPUT_FOLDER = r"C:\Users\aws12\Desktop\Proccesing Datsets ( Tik Tok & Youtube )\بيانات اليوتيوب لمنطقة نجران - بعد المعالجة"   # عدل المسار

TEXT_COLUMN_CANDIDATES = [
    "Text", "text", "Text_Orig", "text_orig", "comment", "Comment", "comments"
]

MIN_WORDS_USEFUL = 3
MIN_ANALYSIS_WORDS = 4
MIN_QUALITY_SCORE = 6
MIN_STRENGTH_SCORE = 7

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# =========================================================
# 2) القوائم
# =========================================================

STOPWORDS = {
    "في", "من", "على", "عن", "الى", "إلى", "او", "أو", "ثم", "لكن", "بس",
    "هذا", "هذه", "هذي", "هنا", "هناك", "هو", "هي", "هم", "انا", "احنا",
    "كان", "كانت", "يكون", "مو", "مش", "مرة", "مره", "جدا", "جداً"
}

NOISE_WORDS = {
    "يفوز", "واو", "ابداع", "فخم", "راقي", "ونعم", "خرافي", "يازينه",
    "حلو", "حلوه", "حلوة", "روعه", "رهيب", "جميل", "جميله", "جميلة",
    "ممتاز", "ابداااع", "يجنن", "ماشاءالله", "تبارك", "الله", "اللهم"
}

VERY_SHORT_OPINIONS = {
    "جميل", "جميله", "جميلة", "رائع", "روعه", "رهيب", "حلو", "حلوه", "حلوة",
    "ممتاز", "يفوز", "فخم", "راقي", "ممتع", "ابداع", "ماشاءالله"
}

AD_WORDS = {
    "واتساب", "تواصل", "احجز", "حجز", "تابعني", "عرض", "عروض", "خصم",
    "للتواصل", "سناب", "انستا", "حسابي", "حسابنا", "للحجز", "اعلان", "إعلان"
}

QUESTION_WORDS = {
    "وين", "كيف", "كم", "هل", "وش", "ايش", "ليش", "متى", "فين", "لوسمحت", "ممكن"
}

INTENT_WORDS = {
    "بروح", "بزوره", "بزورها", "ناوي", "ودي", "ابغى", "ابغا", "لازم",
    "بنروح", "بجي", "اضفته", "اضفتها", "نفسي", "ودي اروح", "ودي ازوره"
}

OPINION_WORDS = {
    "جميل", "جميلة", "رائع", "روعه", "ممتاز", "سيء", "سيئ", "زحمة", "هادئ",
    "هادي", "نظيف", "وصخ", "غالي", "رخيص", "يستاهل", "مايستاهل", "حلو",
    "مزعج", "مريح", "فخم", "بارد", "حر", "حار", "ممتع", "يفشل", "رهيب",
    "كبير", "صغير", "هادية", "ممتعه", "جميله", "رايق", "رايقه", "مناظر",
    "اطلاله", "اطلالة", "تجربه", "تجربة", "خدمات", "اسعار", "نظافه", "نظافة"
}

PLACE_PREFIXES = {
    "قرية", "منتزه", "حديقة", "جبل", "وادي", "شاطئ", "كورنيش", "ممشى",
    "بوليفارد", "متحف", "اكواخ", "كوخ", "مطل", "عين", "سد", "شلال",
    "غابة", "مزرعة", "تلفريك", "بحيرة", "محمية", "واجهة", "جزيرة",
    "قلعة", "حصن", "قصر"
}

GENERIC_PLACE = {
    "فندق", "فنادق", "شقق", "شقة", "منتجع", "منتجعات", "مكان", "المكان",
    "مدينة", "مدينه", "منطقة", "منطقه", "موقع", "معلم", "مزرعة", "حديقة", "منتزه"
}

REGION_CITY_WORDS = {
    "الرياض", "جده", "جدة", "مكه", "مكة", "المدينة", "المدينه", "الطائف", "الطايف",
    "ابها", "أبها", "الباحه", "الباحة", "جازان", "تبوك", "نجران", "حائل", "حايل",
    "القصيم", "الخبر", "الدمام", "الاحساء", "الأحساء", "ينبع", "العلا", "الدرعية",
    "خميس", "مشيط", "المندق", "بلجرشي", "الشفا", "الهدا", "عسير", "السعودية", "المملكة"
}

YOUTUBE_CHANNEL_WORDS = {
    "تصويرك", "تصويركم", "المصور", "مقطعك", "مقطعكم", "قناتك", "قناتكم",
    "ابدعت", "ابدعتوا", "استمر", "استمروا", "الله يعطيك العافيه",
    "الله يعطيك العافية", "شكرا لك", "شكراً لك", "شكرا", "شكراً",
    "مبدع", "ابداعك", "فيديو", "القناة", "اليوتيوب", "نزل", "نزّل",
    "تصوير", "اخراج", "منتاج"
}

# =========================================================
# 3) تنظيف النص
# =========================================================

def normalize_arabic(text):
    if pd.isna(text):
        return ""

    text = str(text)

    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[@#]\S+", " ", text)
    text = re.sub(r"[\u0617-\u061A\u064B-\u0652]", "", text)

    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)

    text = re.sub(r"[A-Za-z0-9]", " ", text)
    text = re.sub(r"[^\u0600-\u06FF\s؟!]", " ", text)

    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

def tokenize(text):
    return [w for w in normalize_arabic(text).split() if len(w.strip()) > 0]

def clean_text(text):
    words = tokenize(text)
    words = [w for w in words if w not in STOPWORDS]
    return " ".join(words).strip()

# =========================================================
# 4) دوال مساعدة
# =========================================================

def is_noise(text):
    words = tokenize(text)
    if len(words) == 0:
        return True
    if len(words) <= 1:
        return True
    if all(w in NOISE_WORDS for w in words):
        return True
    return False

def is_ad(text):
    t = normalize_arabic(text)
    return any(w in t for w in AD_WORDS)

def is_question(text):
    t = normalize_arabic(text)
    return "؟" in t or any(w in t.split() for w in QUESTION_WORDS)

def is_intent(text):
    t = normalize_arabic(text)
    return any(w in t for w in INTENT_WORDS)

def is_opinion(text):
    t = normalize_arabic(text)
    words = t.split()
    return (
        any(w in t for w in OPINION_WORDS) or
        len(words) >= 5
    )

def is_channel_comment(text):
    t = normalize_arabic(text)
    return any(w in t for w in YOUTUBE_CHANNEL_WORDS)

def is_generic_only(words):
    return len(words) > 0 and all(w in GENERIC_PLACE or w in REGION_CITY_WORDS for w in words)

def contains_place(text):
    words = tokenize(text)

    if len(words) < 2:
        return False

    if words[0] in PLACE_PREFIXES and len(words) >= 2:
        if words[1] not in GENERIC_PLACE:
            return True

    if 2 <= len(words) <= 4:
        if not any(w in GENERIC_PLACE for w in words):
            if not any(w in QUESTION_WORDS for w in words):
                if not any(w in AD_WORDS for w in words):
                    return True

    return False

def valid_place_only(text):
    words = tokenize(text)

    if len(words) < 2:
        return False

    if is_generic_only(words):
        return False

    if all(w in GENERIC_PLACE for w in words):
        return False

    if any(w in {"فندق", "فنادق", "شقق", "شقة", "منتجع", "منتجعات"} for w in words):
        return False

    return True

# =========================================================
# 5) تصنيف التعليق
# =========================================================

def classify_comment(text):
    t = normalize_arabic(text)

    if not t:
        return "noise"

    if is_ad(t):
        return "ad"

    if is_channel_comment(t) and not is_opinion(t):
        return "channel_comment"

    if is_noise(t):
        return "noise"

    place_flag = contains_place(t) and valid_place_only(t)
    opinion_flag = is_opinion(t)

    if is_question(t):
        return "question"

    if is_intent(t):
        return "intent"

    if place_flag and opinion_flag:
        return "place_opinion"

    if opinion_flag:
        return "opinion"

    if place_flag:
        return "place_only"

    return "noise"

# =========================================================
# 6) تقييم الجودة والقوة
# =========================================================

def quality_score(text):
    t = normalize_arabic(text)
    words = t.split()

    score = min(len(words), 12)

    if is_opinion(t):
        score += 4
    if contains_place(t):
        score += 2
    if is_question(t):
        score += 1
    if is_intent(t):
        score += 1
    if is_channel_comment(t):
        score -= 3
    if is_ad(t):
        score -= 5
    if is_noise(t):
        score -= 4

    return score

def is_useful_text(text):
    words = tokenize(text)

    if len(words) < MIN_WORDS_USEFUL:
        return False

    short_words = sum(len(w) <= 2 for w in words)
    if short_words > len(words) / 2:
        return False

    return True

def comment_strength(text):
    t = normalize_arabic(text)
    words = t.split()

    score = len(words)

    informative_words = {
        "زحمه", "زحمة", "نظيف", "وصخ", "غالي", "رخيص", "بارد", "حار", "هادئ",
        "هادي", "مريح", "مزعج", "جميل", "رائع", "سيء", "ممتع", "يستاهل", "يفشل",
        "اطلاله", "اطلالة", "مناظر", "تجربه", "تجربة", "خدمات", "اسعار", "نظافه", "نظافة"
    }

    if any(w in t for w in informative_words):
        score += 5

    if len(words) >= 6:
        score += 3

    return score

# =========================================================
# 7) استخراج الأماكن
# =========================================================

def extract_places(text):
    words = tokenize(text)
    places = []

    for i, w in enumerate(words):
        if w in PLACE_PREFIXES:
            phrase = " ".join(words[i:i+4]).strip()
            phrase_words = phrase.split()

            while phrase_words and phrase_words[-1] in REGION_CITY_WORDS:
                phrase_words.pop()

            phrase = " ".join(phrase_words).strip()

            if len(phrase.split()) >= 2:
                if not any(x in {"فندق", "فنادق", "شقق", "شقة", "منتجع"} for x in phrase.split()):
                    places.append(phrase)

    # إضافة دعم محدود للأماكن بدون prefix
    if len(words) == 1 and words[0] not in GENERIC_PLACE and words[0] not in REGION_CITY_WORDS:
        places.append(words[0])

    return places

# =========================================================
# 8) قراءة الملف
# =========================================================

def load_file(file_path):
    if file_path.lower().endswith(".csv"):
        try:
            return pd.read_csv(file_path)
        except UnicodeDecodeError:
            return pd.read_csv(file_path, encoding="utf-8-sig")
    elif file_path.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(file_path)
    else:
        raise ValueError("Unsupported file type")

def detect_text_column(df):
    for col in TEXT_COLUMN_CANDIDATES:
        if col in df.columns:
            return col
    return None

# =========================================================
# 9) معالجة ملف واحد
# =========================================================

def process_file(file_path):
    df = load_file(file_path)

    text_col = detect_text_column(df)
    if text_col is None:
        return {
            "file": os.path.basename(file_path),
            "rows": len(df),
            "analysis_rows": 0,
            "top_place": None,
            "status": "no_text_column"
        }

    df["Text_Normalized"] = df[text_col].apply(normalize_arabic)
    df["Text_Cleaned"] = df[text_col].apply(clean_text)
    df["comment_type"] = df[text_col].apply(classify_comment)
    df["quality_score"] = df[text_col].apply(quality_score)
    df["is_useful"] = df[text_col].apply(is_useful_text)
    df["comment_strength"] = df[text_col].apply(comment_strength)
    df["places"] = df[text_col].apply(extract_places)

    df_analysis = df[
        (df["comment_type"].isin(["opinion", "place_opinion"])) &
        (df["quality_score"] >= MIN_QUALITY_SCORE) &
        (df["comment_strength"] >= MIN_STRENGTH_SCORE) &
        (df["is_useful"] == True)
    ].copy()

    df_analysis = df_analysis[
        (df_analysis["Text_Cleaned"].str.split().str.len() >= MIN_ANALYSIS_WORDS) &
        (~df_analysis["Text_Cleaned"].isin(VERY_SHORT_OPINIONS))
    ].copy()

    counter = Counter()
    for p in df["places"]:
        if isinstance(p, list):
            counter.update(p)

    top_place = counter.most_common(1)[0][0] if counter else None

    name = os.path.splitext(os.path.basename(file_path))[0]
    cleaned_path = os.path.join(OUTPUT_FOLDER, f"{name}_cleaned.xlsx")
    analysis_path = os.path.join(OUTPUT_FOLDER, f"{name}_analysis.xlsx")

    df.to_excel(cleaned_path, index=False)
    df_analysis.to_excel(analysis_path, index=False)

    type_counts = df["comment_type"].value_counts().to_dict()

    return {
        "file": name,
        "rows": len(df),
        "analysis_rows": len(df_analysis),
        "top_place": top_place,
        "opinion_count": type_counts.get("opinion", 0),
        "place_opinion_count": type_counts.get("place_opinion", 0),
        "place_only_count": type_counts.get("place_only", 0),
        "question_count": type_counts.get("question", 0),
        "intent_count": type_counts.get("intent", 0),
        "channel_comment_count": type_counts.get("channel_comment", 0),
        "ad_count": type_counts.get("ad", 0),
        "noise_count": type_counts.get("noise", 0),
        "cleaned_file": cleaned_path,
        "analysis_file": analysis_path,
        "status": "ok"
    }

# =========================================================
# 10) تشغيل على جميع الملفات
# =========================================================

def run_all():
    files = glob.glob(os.path.join(ROOT_FOLDER, "**", "*.xlsx"), recursive=True)
    files += glob.glob(os.path.join(ROOT_FOLDER, "**", "*.xls"), recursive=True)
    files += glob.glob(os.path.join(ROOT_FOLDER, "**", "*.csv"), recursive=True)

    results = []

    for f in files:
        print("Processing:", f)
        try:
            res = process_file(f)
            results.append(res)
        except Exception as e:
            results.append({
                "file": os.path.basename(f),
                "rows": 0,
                "analysis_rows": 0,
                "top_place": None,
                "opinion_count": 0,
                "place_opinion_count": 0,
                "place_only_count": 0,
                "question_count": 0,
                "intent_count": 0,
                "channel_comment_count": 0,
                "ad_count": 0,
                "noise_count": 0,
                "cleaned_file": None,
                "analysis_file": None,
                "status": f"error: {str(e)}"
            })

    summary = pd.DataFrame(results)
    summary.to_excel(os.path.join(OUTPUT_FOLDER, "youtube_summary.xlsx"), index=False)

    print("Finished 🚀")

# =========================================================
# تشغيل
# =========================================================

run_all()

Processing: C:\Users\aws12\Desktop\Proccesing Datsets ( Tik Tok & Youtube )\YouTube Datasets - Najran\Najran Vedio Comments 1_textready.xlsx
Processing: C:\Users\aws12\Desktop\Proccesing Datsets ( Tik Tok & Youtube )\YouTube Datasets - Najran\Najran Vedio Comments 2_textready.xlsx
Processing: C:\Users\aws12\Desktop\Proccesing Datsets ( Tik Tok & Youtube )\YouTube Datasets - Najran\Najran Vedio Comments 3_textready.xlsx
Processing: C:\Users\aws12\Desktop\Proccesing Datsets ( Tik Tok & Youtube )\YouTube Datasets - Najran\Najran Vedio Comments 4_textready.xlsx
Processing: C:\Users\aws12\Desktop\Proccesing Datsets ( Tik Tok & Youtube )\YouTube Datasets - Najran\Najran Vedio Comments 5_textready.xlsx
Processing: C:\Users\aws12\Desktop\Proccesing Datsets ( Tik Tok & Youtube )\YouTube Datasets - Najran\Najran Vedio Comments 6_textready.xlsx
Processing: C:\Users\aws12\Desktop\Proccesing Datsets ( Tik Tok & Youtube )\YouTube Datasets - Najran\Najran Vedio Comments 7_textready.xlsx
Processing: C